In [ ]:
import polars as pl
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
from dotenv import dotenv_values

In [22]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
from dotenv import dotenv_values


In [28]:
config = dotenv_values(".env")


In [26]:
client = TypeSafeClient(api_key=config["TYPESAFE_API_KEY"])


In [53]:
from typesafe_sdk import Choice, TypeSafeClient


def klassifiser_kamp(lag1: str, lag2: str, oppsummering: str):
    with TypeSafeClient(api_key=config["TYPESAFE_API_KEY"]) as client:
        response = client.system_one(
            state={
                "lag1": lag1,
                "lag2": lag2,
                "oppsummering": oppsummering,
            },
            questions={
                "resultat": Choice(
                    instructions=(
                        "Bestem sluttresultatet i fotballkampen basert på "
                        "oppsummeringen. Velg hvilket lag som vant, eller uavgjort. "
                        "Ikke vurder hvem som spilte best eller hadde flest sjanser. "
                        "Klassifiser kun kampens resultat."
                    ),
                    criteria={
                        f"{lag1}_vant": (
                            f"{lag1} vant kampen mot {lag2}. "
                            f"Oppsummeringen viser at {lag1} scoret flere mål, "
                            f"eller eksplisitt slo/beseiret {lag2}."
                        ),
                        f"{lag2}_vant": (
                            f"{lag2} vant kampen mot {lag1}. "
                            f"Oppsummeringen viser at {lag2} scoret flere mål, "
                            f"eller eksplisitt slo/beseiret {lag1}."
                        ),
                        "uavgjort": (
                            "Kampen endte uavgjort. Lagene hadde like mange mål "
                            "da kampen var ferdig."
                        ),
                    },
                )
            },
        )

    resultat = response.choices["resultat"]

    return resultat.choice, resultat.confidence,resultat.probabilities

In [64]:
referat = """Utrolig nok ebber det ut uten scoringer her i Sandefjord! Det vil nok hjemmelaget være mest misfornøyd med for de skapte et vel av muligheter, og hadde også en ball i tverrligger da Dunsby headet. Ellers parerte Dyngeland det som kom hans vei. På motsatt side stod Hadaya på rett plass da Brann fikk sine to sjanser i andre omgang. Dermed blir det ett poeng til hvert av lagene etter en fartsfylt forestilling! 0-0 kamper kan være artige de også!"""

In [65]:
resultat, confidence, probabilities = klassifiser_kamp("Brann", "Sandefjord", referat)

In [66]:
resultat, confidence, probabilities

('uavgjort', 1.0, {'Brann_vant': 0.0, 'Sandefjord_vant': 0.0, 'uavgjort': 1.0})

In [ ]:
def rut_sporsmal(sporsmal: str):
    with TypeSafeClient(api_key=config["TYPESAFE_API_KEY"]) as client:
        response = client.system_one(
            state={"sporsmal": sporsmal},
            questions={
                "rute": Choice(
                    instructions=(
                        "Bestem hvilken datakilde som best besvarer brukerens spørsmål. "
                        "Velg kun én rute basert på hva spørsmålet krever."
                    ),
                    criteria={
                        "strukturert": (
                            "Spørsmålet kan besvares med en SQL-spørring mot tabeller: "
                            "kampresultater, tabellplassering, statistikk, tellinger, "
                            "summer, gjennomsnitt, rangeringer eller filtrering på tall, "
                            "datoer og lag."
                        ),
                        "rag": (
                            "Spørsmålet krever å slå opp og tolke fritekst, som "
                            "kampreferater, beskrivelser, kommentarer eller forklaringer "
                            "om hva som skjedde i en kamp."
                        ),
                    },
                )
            },
        )

    rute = response.choices["rute"]
    return rute.choice, rute.confidence, rute.probabilities
